In [1]:
from autogluon.tabular import TabularDataset, TabularPredictor
import pandas as pd

In [2]:
save_path = "../AutogluonModels/TabularPredictor"
df = pd.read_csv("../data/processed/datas_cleaned.csv")
df.head()

,year,region,size_estabish,value,size_rank
0,2542,Bangkok,<60,297061700.0,1
1,2544,Bangkok,<60,316149600.0,1
2,2546,Bangkok,<60,292475000.0,1
3,2548,Bangkok,<60,318711400.0,1
4,2550,Bangkok,<60,443951100.0,1


In [3]:
train_data = TabularDataset(df[df["year"] <= 2554].drop(columns=["size_estabish"]))
test_data = TabularDataset(df[df["year"] >= 2556].drop(columns=["size_estabish"]))

In [4]:
train_data["value"].describe()

count    1.050000e+02
mean     5.046461e+09
std      7.965187e+09
min      2.223934e+08
25%      8.673075e+08
50%      1.726217e+09
75%      5.033558e+09
max      3.997165e+10
Name: value, dtype: float64

In [5]:
predictor = TabularPredictor(
    label="value", problem_type="regression", path=save_path
).fit(
    train_data,
    presets="good_quality",
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          20
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       6.91 GB / 15.73 GB (44.0%)
Disk Space Avail:   636.31 GB / 953.85 GB (66.7%)
Presets specified: ['good_quality']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack 

(_ray_fit pid=27028) [1000]	valid_set's rmse: 3.12965e+09


(_dystack pid=24440) 	-7091822093.6443	 = Validation score   (-root_mean_squared_error)
(_dystack pid=24440) 	1.08s	 = Training   runtime
(_dystack pid=24440) 	0.02s	 = Validation runtime
(_dystack pid=24440) Fitting model: LightGBM_BAG_L1 ... Training model for up to 588.30s of the 886.26s of remaining time.
(_dystack pid=24440) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.34%)
(_dystack pid=24440) 	-6544775955.7193	 = Validation score   (-root_mean_squared_error)
(_dystack pid=24440) 	0.46s	 = Training   runtime
(_dystack pid=24440) 	0.02s	 = Validation runtime
(_dystack pid=24440) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 584.03s of the 881.99s of remaining time.
(_dystack pid=24440) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=20, gpus=0
(_dystack pid=24440) 	-3084016045.3628	 = Validation score   (-root_mean_squared_error)
(_dystack pid=24440) 	0.51s

(_ray_fit pid=21936) [1000]	valid_set's rmse: 1.80819e+09 [repeated 12x across cluster]


(_dystack pid=24440) 	-3470721042.7922	 = Validation score   (-root_mean_squared_error)
(_dystack pid=24440) 	0.65s	 = Training   runtime
(_dystack pid=24440) 	0.02s	 = Validation runtime
(_ray_fit pid=15264) d:\Mini Project\Term Project\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning. [repeated 7x across cluster]
(_ray_fit pid=15264)   warnings.warn( [repeated 7x across cluster]
(_dystack pid=24440) Fitting model: WeightedEnsemble_L3 ... Training model for up to 360.00s of the 507.46s of remaining time.
(_dystack pid=24440) 	Fitting 1 model on all data | Fitting with cpus=20, gpus=0, mem=0.0/4.5 GB
(_dystack pid=24440) 	Ensemble Weights: {'NeuralNetFastAI_BAG_L2': 0.529, 'XGBoost_BAG_L1': 0.412, 'ExtraTreesMSE_BAG_L2': 0.059}
(_dystack pid=24440) 	-2550151482.25	 = Validation score   (-root_mean_squa

In [6]:
predictor.evaluate(test_data)

{'root_mean_squared_error': np.float64(-7722535144.447288),
 'mean_squared_error': -5.963754905722348e+19,
 'mean_absolute_error': -4913063376.666667,
 'r2': 0.6029729238362012,
 'pearsonr': 0.838044041972232,
 'median_absolute_error': -2478005030.0}

In [7]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForestMSE_BAG_L1_FULL,-7.421274e+09,NaN,root_mean_squared_error,0.055053,0.043004,0.557643,0.055053,0.043004,0.557643,1,True,13
1,RandomForestMSE_BAG_L1,-7.421274e+09,-3.221726e+09,root_mean_squared_error,0.057033,0.043004,0.557643,0.057033,0.043004,0.557643,1,True,3
2,ExtraTreesMSE_BAG_L1,-7.428290e+09,-3.271058e+09,root_mean_squared_error,0.042032,0.045742,0.343870,0.042032,0.045742,0.343870,1,True,5
3,ExtraTreesMSE_BAG_L1_FULL,-7.428290e+09,NaN,root_mean_squared_error,0.063544,0.045742,0.343870,0.063544,0.045742,0.343870,1,True,15
4,WeightedEnsemble_L2_FULL,-7.722535e+09,NaN,root_mean_squared_error,0.080053,NaN,27.672735,0.003000,NaN,0.005999,2,True,20
5,XGBoost_BAG_L1_FULL,-8.087801e+09,NaN,root_mean_squared_error,0.007999,NaN,0.133164,0.007999,NaN,0.133164,1,True,17
6,CatBoost_BAG_L1_FULL,-8.579163e+09,NaN,root_mean_squared_error,0.005999,NaN,24.509723,0.005999,NaN,24.509723,1,True,14
7,NeuralNetTorch_BAG_L1_FULL,-8.598606e+09,NaN,root_mean_squared_error,0.008001,NaN,2.466206,0.008001,NaN,2.466206,1,True,18
8,NeuralNetFastAI_BAG_L1_FULL,-8.614052e+09,NaN,root_mean_squared_error,0.037243,NaN,0.474050,0.037243,NaN,0.474050,1,True,16
9,LightGBM_BAG_L1_FULL,-1.112604e+10,NaN,root_mean_squared_error,0.031004,NaN,0.265540,0.031004,NaN,0.265540,1,True,12


In [9]:
predictor.predict(test_data)

7      4.294946e+09
8      4.403822e+09
16     5.713501e+09
17     5.868390e+09
25     2.381346e+10
26     2.375378e+10
34     9.607648e+09
35     9.821832e+09
43     7.862944e+09
44     8.141699e+09
52     1.504830e+10
53     1.519627e+10
61     6.094908e+09
62     6.342513e+09
70     4.701615e+09
71     5.026607e+09
79     6.364384e+09
80     6.683910e+09
88     3.822315e+09
89     3.983748e+09
97     3.510669e+09
98     3.694706e+09
106    3.684732e+09
107    3.812152e+09
115    2.114438e+10
116    2.138064e+10
124    1.315743e+10
125    1.340286e+10
133    1.893995e+10
134    1.901410e+10
Name: value, dtype: float32